# dj-server Demo

This notebook demonstrates how to use the `dj-server` package, a DataJoint-powered Flask server for managing and querying neuroscience experiment data.

**Installation** (run once in your conda environment):
```bash
pip install -e /path/to/datajoint-1
```

**Prerequisites**:
- Docker Desktop running (for MySQL database containers)
- `dj-server` installed in the active Python environment

In [ ]:
import requests
import json
import subprocess
import time
import os

## 1. Start the Server

`dj-server` runs as a Flask server on port 5000. You can start it as a background process from the notebook, or run `dj-server` in a separate terminal.

In [ ]:
# Start dj-server in background
server_process = subprocess.Popen(
    ["dj-server"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(3)  # Wait for server to initialize

BASE_URL = "http://127.0.0.1:5000"
print("Server started!" if server_process.poll() is None else "Server failed to start")

## 2. Database Setup

Each database runs as an isolated Docker container with MySQL. The workflow is:
1. Set the directory where databases are stored
2. Create a new database (downloads a docker-compose.yaml)
3. Start the database (spins up a Docker container)
4. Connect to the database

In [ ]:
# Point to where databases are stored
db_dir = os.path.expanduser("~/databases")
os.makedirs(db_dir, exist_ok=True)

resp = requests.post(f"{BASE_URL}/init/set-database-directory", json={"dir": db_dir})
print(resp.json())

In [ ]:
# List existing databases
resp = requests.get(f"{BASE_URL}/init/list-databases")
print("Existing databases:", resp.json())

# Create a new database (downloads docker-compose.yaml)
resp = requests.post(f"{BASE_URL}/init/create-database", json={"name": "demo_db"})
print(resp.json())

# Start the database (requires Docker Desktop running)
resp = requests.post(f"{BASE_URL}/init/start-database", json={"name": "demo_db"})
print(resp.json())

# Connect to the database (may take a moment on first start)
time.sleep(5)  # Wait for MySQL container to initialize
resp = requests.post(f"{BASE_URL}/init/connect-database", json={"name": "demo_db"})
print(resp.json())

## 3. Set User & Add Data

Set a username (used for tagging), then populate the database from your experiment files.

Data directories should contain:
- **data**: `.h5` files (HDF5 experiment recordings)
- **meta**: `.json` files (metadata, generated by `parse_data.py`)
- **tags**: `.json` files (user annotations, can be empty `{}`)

In [ ]:
# Set your username
resp = requests.post(f"{BASE_URL}/user/set-user", json={"user": "demo_user"})
print(resp.json())

In [ ]:
# Check if database has any experiments
resp = requests.get(f"{BASE_URL}/pop/is-empty")
print(resp.json())

# Add data from directories (uncomment and adjust paths to your data)
# resp = requests.post(f"{BASE_URL}/pop/add-data", json={
#     "data_dir": "/Volumes/data/datajoint_testbed/data",
#     "meta_dir": "/Volumes/data/datajoint_testbed/meta",
#     "tags_dir": "/Volumes/data/datajoint_testbed/tags"
# })
# print(resp.json())

# Poll until data addition is complete
# while requests.get(f"{BASE_URL}/pop/is-adding").json().get("adding"):
#     time.sleep(5)
#     print("Still adding data...")
# print("Done!")

## 4. Querying Data

The query system uses a hierarchical structure matching the experiment data model:

```
Experiment > Animal > Preparation > Cell > EpochGroup > EpochBlock > Epoch > Response/Stimulus
```

Each level supports nested AND/OR/NOT conditions. Set a level to `None` for no filtering.

In [ ]:
# Get available hierarchy levels and their queryable fields
resp = requests.get(f"{BASE_URL}/query/get-levels-and-fields")
data = resp.json()

print("Hierarchy levels:", data["levels"])
print("\nExample fields for 'experiment':")
for field, ftype in data["fields"]["experiment"]:
    print(f"  {field}: {ftype}")

In [ ]:
# Build a query object - empty conditions (None) returns all data
query_obj = {
    "experiment": None,
    "animal": None,
    "preparation": None,
    "cell": None,
    "epoch_group": None,
    "epoch_block": None,
    "epoch": None,
}

# Execute the query, skipping intermediate levels for faster results
resp = requests.post(f"{BASE_URL}/query/execute-query", json={
    "query_obj": query_obj,
    "exclude_levels": ["animal", "preparation", "cell"]
})

results = resp.json()
if "results" in results:
    print(f"Got {len(results['results'])} top-level results")
    print(json.dumps(results["results"][0], indent=2, default=str))
else:
    print(results)

In [ ]:
# Example: query with conditions
# Find all epoch blocks where protocol name contains "noise"
filtered_query = {
    "experiment": None,
    "animal": None,
    "preparation": None,
    "cell": None,
    "epoch_group": None,
    "epoch_block": {
        "AND": [
            {"COND": {"type": "PARAM", "value": "protocol_name like '%noise%'"}}
        ]
    },
    "epoch": None,
}

# resp = requests.post(f"{BASE_URL}/query/execute-query", json={
#     "query_obj": filtered_query,
#     "exclude_levels": ["animal", "preparation", "cell"]
# })
# print(resp.json())

## 5. Results: Metadata, Tags, and Visualization

After executing a query, you can:
- Fetch full metadata for any item
- Add/delete tags (associated with your username)
- Push/pull/reset tags to sync with other users via the filesystem
- Get visualizations (spike traces for single-cell, spike histograms for MEA)

In [ ]:
# Get metadata for a specific item (uncomment after populating data)
# resp = requests.post(f"{BASE_URL}/results/get-metadata", json={
#     "level": "experiment",
#     "id": 1
# })
# print(json.dumps(resp.json(), indent=2, default=str))

In [ ]:
# Tag operations
# Add a tag to selected items (format: "experiment_id-level-id")
# requests.post(f"{BASE_URL}/results/add-tags", json={
#     "ids": ["1-epoch-42", "1-epoch-43"],
#     "tag": "interesting"
# })

# Push your tags to the filesystem for sharing
# requests.post(f"{BASE_URL}/results/push-tags", json={"experiment_ids": [1]})

# Pull other users' tags from the filesystem
# requests.post(f"{BASE_URL}/results/pull-tags", json={"experiment_ids": [1]})

In [ ]:
# Save a query for later reuse
# requests.post(f"{BASE_URL}/query/add-saved-query", json={
#     "query_name": "all_experiments",
#     "query_obj": query_obj
# })

# List saved queries
# resp = requests.get(f"{BASE_URL}/query/get-saved-queries")
# print(resp.json())

# Download results as JSON
# requests.post(f"{BASE_URL}/results/download-results", json={
#     "exclude_levels": False,
#     "include_meta": True
# })

## 6. Direct Python Import

You can also import `dj_server` modules directly for programmatic use without going through the REST API.

In [ ]:
from dj_server.helpers.utils import table_arr, fields
from dj_server.helpers.db_lifecycle import create_database, start_database

print("Table hierarchy:", table_arr)
print("\nExperiment field mappings (db_field -> json_key):")
for db_field, json_key in fields['experiment']:
    print(f"  {db_field} <- {json_key}")

## 7. Cleanup

In [ ]:
# Stop the database when done
# requests.post(f"{BASE_URL}/init/stop-database", json={"name": "demo_db"})

# Terminate the server process
server_process.terminate()
print("Server stopped")